<a href="https://colab.research.google.com/github/lennalena2555-prog/Business-Intelligence-assignment/blob/main/Pretrained_transformer_using_hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y torchvision torchaudio
!pip install -q transformers datasets evaluate accelerate scikit-learn

# IMDb SENTIMENT ANALYSIS USING DISTILBERT
# Dataset:
# Q-b1t/IMDB-Dataset-of-50K-Movie-Reviews-Backup


# 2. IMPORT LIBRARIES

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# 3. LOAD THE IMDb DATASET

print("Loading IMDb dataset...")

ds = load_dataset(
    "Q-b1t/IMDB-Dataset-of-50K-Movie-Reviews-Backup"
)

print("\nDataset loaded successfully!")
print(ds)

# 4. INSPECT DATASET STRUCTURE

print("\nAvailable dataset splits:")

for split in ds.keys():
    print(
        f"{split}: {len(ds[split])} records"
    )

print("\nDataset columns:")

for split in ds.keys():
    print(
        f"{split}: {ds[split].column_names}"
    )

# Display first record
first_split = list(ds.keys())[0]

print("\nFirst record:")
print(ds[first_split][0])

# 5. IDENTIFY TRAINING AND TEST DATA

# If the dataset already contains train/test splits,
# use them directly.

if "train" in ds and "test" in ds:

    train_dataset = ds["train"]
    test_dataset = ds["test"]

# If only one split exists, divide it into train/test.
else:

    available_split = list(ds.keys())[0]

    full_dataset = ds[available_split]

    split_dataset = full_dataset.train_test_split(
        test_size=0.2,
        seed=42
    )

    train_dataset = split_dataset["train"]
    test_dataset = split_dataset["test"]

print("\nTraining samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

# 6. IDENTIFY TEXT AND LABEL COLUMNS

print("\nColumns in dataset:")
print(train_dataset.column_names)

# The common IMDb dataset uses:
# text  -> movie review
# label -> sentiment

if "text" in train_dataset.column_names:

    text_column = "text"

elif "review" in train_dataset.column_names:

    text_column = "review"

else:

    # Automatically identify a string column
    text_column = None

    for column in train_dataset.column_names:

        if train_dataset.features[column].dtype == "string":

            text_column = column
            break

    if text_column is None:
        raise ValueError(
            "Could not identify the review/text column."
        )

print("\nText column:", text_column)

# Identify label column

if "label" in train_dataset.column_names:

    label_column = "label"

elif "sentiment" in train_dataset.column_names:

    label_column = "sentiment"

else:

    raise ValueError(
        "Could not identify the sentiment/label column."
    )

print("Label column:", label_column)

# ============================================================
# 7. CONVERT TEXT LABELS TO NUMERIC LABELS IF NECESSARY
# ============================================================

print("\nChecking labels...")
print(train_dataset.features[label_column])
print("Example label:", train_dataset[0][label_column])

if train_dataset.features[label_column].dtype == "string":

    def convert_labels(example):
        raw_label = str(example[label_column]).lower().strip()
        if raw_label in ["negative", "neg", "0"]:
            val = 0
        elif raw_label in ["positive", "pos", "1"]:
            val = 1
        else:
            raise ValueError(f"Unknown label: {raw_label}")

        return {"target_label": val}

    # Map to new integer column, then drop old string column
    train_dataset = train_dataset.map(convert_labels).remove_columns([label_column]).rename_column("target_label", "label")
    test_dataset = test_dataset.map(convert_labels).remove_columns([label_column]).rename_column("target_label", "label")
    label_column = "label"

elif label_column != "label":
    train_dataset = train_dataset.rename_column(label_column, "label")
    test_dataset = test_dataset.rename_column(label_column, "label")
    label_column = "label"

print("\nFinal label column:", label_column)

# 8. DISPLAY CLASS DISTRIBUTION

print("\nTraining label distribution:")

print(
    pd.Series(
        train_dataset["label"]
    ).value_counts()
)

print("\nTesting label distribution:")

print(
    pd.Series(
        test_dataset["label"]
    ).value_counts()
)

# 9. LOAD PRETRAINED DISTILBERT TOKENIZER

model_name = "distilbert-base-uncased"

print(
    "\nLoading pretrained model:",
    model_name
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

# 10. TOKENIZE THE DATASET

def tokenize_function(examples):

    return tokenizer(
        examples[text_column],
        padding="max_length",
        truncation=True,
        max_length=256
    )

print("\nTokenizing training dataset...")

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenizing testing dataset...")

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print("\nTokenization completed.")

# 11. REMOVE ORIGINAL TEXT COLUMN

# Keep only the columns required by the model.

columns_to_remove = [
    column
    for column in tokenized_train.column_names
    if column not in [
        "input_ids",
        "attention_mask",
        "label"
    ]
]

tokenized_train = tokenized_train.remove_columns(
    columns_to_remove
)

columns_to_remove = [
    column
    for column in tokenized_test.column_names
    if column not in [
        "input_ids",
        "attention_mask",
        "label"
    ]
]

tokenized_test = tokenized_test.remove_columns(
    columns_to_remove
)


# 12. SET PYTORCH FORMAT

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print("\nTokenized training columns:")
print(tokenized_train.column_names)

# 13. LOAD PRETRAINED DISTILBERT MODEL

print("\nLoading DistilBERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Define label names

model.config.id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

model.config.label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}


# 14. DEFINE EVALUATION METRICS

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Convert logits into predicted class
    predictions = np.argmax(
        logits,
        axis=-1
    )

    # Calculate precision, recall and F1
    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            zero_division=0
        )
    )

    # Calculate accuracy
    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# 15. CHECK GPU AVAILABILITY

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nTraining device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "WARNING: GPU not detected."
    )


# 16. DEFINE TRAINING ARGUMENTS

training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=2,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_steps=100,

    report_to="none",

    fp16=torch.cuda.is_available()
)


# 17. CREATE HUGGING FACE TRAINER

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)


# 18. FINE-TUNE DISTILBERT

print("\n")
print("=" * 60)
print("STARTING MODEL TRAINING")
print("=" * 60)

trainer.train()

# 19. EVALUATE THE MODEL

print("\n")
print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

evaluation_results = trainer.evaluate()

for key, value in evaluation_results.items():

    if isinstance(value, float):

        print(
            f"{key}: {value:.4f}"
        )

    else:

        print(
            f"{key}: {value}"
        )

# 20. MAKE PREDICTIONS ON THE TEST DATA

print("\nGenerating predictions...")

predictions_output = trainer.predict(
    tokenized_test
)

predicted_labels = np.argmax(
    predictions_output.predictions,
    axis=-1
)

true_labels = predictions_output.label_ids

# 21. CLASSIFICATION REPORT

print("\n")
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[
            "Negative",
            "Positive"
        ],
        digits=4
    )
)

# 22. CONFUSION MATRIX

print("\n")
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print(cm)

print("\nInterpretation:")
print(
    "Rows represent the actual classes and "
    "columns represent the predicted classes."
)

# 23. TEST MODEL WITH NEW MOVIE REVIEWS

reviews = [

    "This movie was absolutely fantastic. "
    "The acting was excellent and the story was amazing.",

    "The movie was terrible and extremely boring. "
    "I would not recommend it to anyone.",

    "An excellent film with great performances "
    "and a brilliant storyline.",

    "I hated this movie. "
    "It was a complete waste of time."
]

# Move model to available device

model.to(device)

# Tokenize new reviews

inputs = tokenizer(
    reviews,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

# Move tensors to device

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Disable gradient calculation

with torch.no_grad():

    outputs = model(**inputs)

# Get predictions

new_predictions = torch.argmax(
    outputs.logits,
    dim=-1
)

# Display predictions

print("\n")
print("=" * 60)
print("NEW REVIEW SENTIMENT PREDICTIONS")
print("=" * 60)

for review, prediction in zip(
    reviews,
    new_predictions
):

    if prediction.item() == 1:

        sentiment = "POSITIVE"

    else:

        sentiment = "NEGATIVE"

    print("\nReview:")
    print(review)

    print(
        "Predicted Sentiment:",
        sentiment
    )

# 24. SAVE THE FINE-TUNED MODEL

save_directory = "./imdb-distilbert"

model.save_pretrained(
    save_directory
)

tokenizer.save_pretrained(
    save_directory
)

print("\n")
print("=" * 60)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 60)

print(
    f"Model location: {save_directory}"
)


# 25. FINAL SUMMARY

print("\n")
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print(
    f"Training samples: {len(train_dataset)}"
)

print(
    f"Testing samples: {len(test_dataset)}"
)

print(
    f"Model: {model_name}"
)

print(
    f"Maximum sequence length: 256"
)

print(
    f"Training epochs: 2"
)

print(
    f"Learning rate: 2e-5"
)

print(
    f"Accuracy: {evaluation_results['eval_accuracy']:.4f}"
)

print(
    f"Precision: {evaluation_results['eval_precision']:.4f}"
)

print(
    f"Recall: {evaluation_results['eval_recall']:.4f}"
)

print(
    f"F1-score: {evaluation_results['eval_f1']:.4f}"
)

print("\nExperiment completed successfully!")

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
Loading IMDb dataset...


README.md:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

archive.zip: reconstructing file:   0%|          |  0.00B / 27.0MB            

archive.zip: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.190581,0.245559,2,0.920800,0.912896,0.928065,0.920418


eval_loss: 0.2456
eval_accuracy: 0.9208
eval_precision: 0.9129
eval_recall: 0.9281
eval_f1: 0.9204

Generating predictions...




CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Negative     0.9288    0.9137    0.9212      5065
    Positive     0.9129    0.9281    0.9204      4935

    accuracy                         0.9208     10000
   macro avg     0.9208    0.9209    0.9208     10000
weighted avg     0.9209    0.9208    0.9208     10000



CONFUSION MATRIX
[[4628  437]
 [ 355 4580]]

Interpretation:
Rows represent the actual classes and columns represent the predicted classes.


NEW REVIEW SENTIMENT PREDICTIONS

Review:
This movie was absolutely fantastic. The acting was excellent and the story was amazing.
Predicted Sentiment: POSITIVE

Review:
The movie was terrible and extremely boring. I would not recommend it to anyone.
Predicted Sentiment: NEGATIVE

Review:
An excellent film with great performances and a brilliant storyline.
Predicted Sentiment: POSITIVE

Review:
I hated this movie. It was a complete waste of time.
Predicted Sentiment: NEGATIVE


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]



MODEL SAVED SUCCESSFULLY
Model location: ./imdb-distilbert


FINAL SUMMARY
Training samples: 40000
Testing samples: 10000
Model: distilbert-base-uncased
Maximum sequence length: 256
Training epochs: 2
Learning rate: 2e-5
Accuracy: 0.9208
Precision: 0.9129
Recall: 0.9281
F1-score: 0.9204

Experiment completed successfully!
